In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

project_dir = Path("~/Developer/project/monster-hack").expanduser()
data_dir = project_dir / "Dataset" / "Dataset"
train_dir = data_dir / "Training"

lms_path = train_dir / "la_mmh_training_lms_item_activity_by_session_fact.csv"
video_path = train_dir / "la_mmh_training_video_activity_by_view_fact.csv"
hunt_answers_path = project_dir / "HUNT_P_IDs.csv"

for file_path in [lms_path, video_path, hunt_answers_path]:
    if not file_path.is_file():
        raise FileNotFoundError(f"Missing file: {file_path}")


In [2]:
hunt_answers = pd.read_csv(
    hunt_answers_path,
    dtype={"P_ID": "string"},
)

hunt_answers["monster_key"] = (
    hunt_answers["Monster"]
    .astype("string")
    .str.lower()
    .str.replace(r"[^a-z]", "", regex=True)
)

vampire_answers = (
    hunt_answers.loc[
        hunt_answers["monster_key"].eq("vampire"),
        ["P_ID"],
    ]
    .dropna()
    .drop_duplicates()
    .sort_values("P_ID")
    .reset_index(drop=True)
)

print(f"Loaded {len(vampire_answers):,} unique Vampire answers")
display(vampire_answers.head())


Loaded 99 unique Vampire answers


,P_ID
0,P00000775
1,P00000806
2,P00000843
3,P00000974
4,P00001103


In [3]:
# We will inspect activity only for the answer IDs to keep the evidence manageable.
vampire_ids = set(vampire_answers["P_ID"])
print(f"Prepared {len(vampire_ids):,} IDs for activity evidence")


Prepared 99 IDs for activity evidence


In [4]:
def get_vampire_activity_evidence(path, source_name, vampire_ids, chunksize=200_000):
    night_parts = []
    day_parts = []
    rows_scanned = 0

    for chunk in pd.read_csv(
        path,
        usecols=["P_ID", "TIMESTAMP"],
        dtype={"P_ID": "string", "TIMESTAMP": "string"},
        chunksize=chunksize,
    ):
        rows_scanned += len(chunk)
        chunk = chunk.loc[chunk["P_ID"].isin(vampire_ids)].copy()
        if chunk.empty:
            continue

        chunk["timestamp"] = pd.to_datetime(
            chunk["TIMESTAMP"],
            utc=True,
            errors="coerce",
        )
        chunk = chunk.dropna(subset=["timestamp"])
        hour = chunk["timestamp"].dt.hour + chunk["timestamp"].dt.minute / 60
        chunk["activity_date"] = chunk["timestamp"].dt.floor("D").dt.tz_localize(None)

        night_parts.append(
            chunk.loc[(hour > 0) & (hour < 5), ["P_ID", "activity_date"]]
            .drop_duplicates()
            .assign(source=source_name)
        )
        day_parts.append(
            chunk.loc[(hour > 5) & (hour < 24), ["P_ID", "activity_date"]]
            .drop_duplicates()
            .assign(source=source_name)
        )

    night = pd.concat(night_parts, ignore_index=True) if night_parts else pd.DataFrame(columns=["P_ID", "activity_date", "source"])
    day = pd.concat(day_parts, ignore_index=True) if day_parts else pd.DataFrame(columns=["P_ID", "activity_date", "source"])
    print(f"{source_name}: scanned {rows_scanned:,} rows")
    return night, day


In [5]:
lms_night, lms_day = get_vampire_activity_evidence(lms_path, "LMS", vampire_ids)
video_night, video_day = get_vampire_activity_evidence(video_path, "Video", vampire_ids)

night_activity = pd.concat([lms_night, video_night], ignore_index=True).drop_duplicates(["P_ID", "activity_date"])
day_activity = pd.concat([lms_day, video_day], ignore_index=True).drop_duplicates(["P_ID", "activity_date"])

night_counts = night_activity.groupby("P_ID").agg(
    night_active_days=("activity_date", "nunique"),
    first_night_date=("activity_date", "min"),
    last_night_date=("activity_date", "max"),
)
day_counts = day_activity.groupby("P_ID").agg(
    day_active_days=("activity_date", "nunique"),
    first_day_date=("activity_date", "min"),
    last_day_date=("activity_date", "max"),
)

vampire_evidence = (
    vampire_answers.merge(night_counts, on="P_ID", how="left")
    .merge(day_counts, on="P_ID", how="left")
    .fillna({"night_active_days": 0, "day_active_days": 0})
)

print(f"Attached activity evidence for {vampire_evidence['P_ID'].nunique():,} answer IDs")
display(vampire_evidence.head(20))


LMS: scanned 7,857,424 rows
Video: scanned 1,145,358 rows
Attached activity evidence for 99 answer IDs


,P_ID,night_active_days,first_night_date,last_night_date,day_active_days,first_day_date,last_day_date
0,P00000775,167,2023-10-02,2024-06-13,157,2023-10-02,2024-06-13
1,P00000806,279,2023-10-03,2024-09-20,237,2023-10-05,2024-09-04
2,P00000843,8,2023-10-05,2024-06-04,235,2023-10-04,2024-08-16
3,P00000974,183,2023-10-03,2024-06-14,168,2023-10-03,2024-06-14
4,P00001103,209,2023-10-03,2024-08-06,188,2023-10-03,2024-07-20
5,P00001130,206,2023-10-02,2024-09-18,184,2023-10-02,2024-06-25
6,P00001218,209,2023-10-02,2024-08-13,180,2023-10-04,2024-08-01
7,P00001243,195,2023-10-04,2024-09-03,174,2023-10-05,2024-08-26
8,P00001812,229,2023-10-06,2024-08-27,202,2023-10-06,2024-08-27
9,P00002030,235,2023-10-04,2024-08-12,199,2023-10-04,2024-08-11


In [6]:
vampire_submission = vampire_answers.copy()
vampire_submission.insert(0, "Monster Type", "Vampire")

print(f"Prepared {len(vampire_submission):,} unique Vampire IDs")
display(vampire_submission)


Prepared 99 unique Vampire IDs


,Monster Type,P_ID
0,Vampire,P00000775
1,Vampire,P00000806
2,Vampire,P00000843
3,Vampire,P00000974
4,Vampire,P00001103
...,...,...
94,Vampire,P00021323
95,Vampire,P00021355
96,Vampire,P00021362
97,Vampire,P00021609


In [7]:
# The CSV is the authoritative HUNT answer list; keep every listed Vampire ID in the submission.
answer_ids = set(vampire_answers["P_ID"])
submission_ids = set(vampire_submission["P_ID"])
assert answer_ids == submission_ids
assert not vampire_submission["P_ID"].duplicated().any()

print(f"Official Vampire IDs retained in submission: {len(vampire_submission):,}")

# This is a separate diagnostic of the time-window interpretation.
# It does not remove IDs from the official answer-based submission.
rule_ids = set(
    vampire_evidence.loc[
        (vampire_evidence["night_active_days"] > 21)
        & (vampire_evidence["night_active_days"] > vampire_evidence["day_active_days"]),
        "P_ID",
    ]
)
rule_check = pd.DataFrame({
    "check": ["Official answer IDs", "Also matching implemented timestamp rule", "Needs timestamp-rule review"],
    "count": [len(answer_ids), len(answer_ids & rule_ids), len(answer_ids - rule_ids)],
})
display(rule_check)
display(vampire_submission)


Official Vampire IDs retained in submission: 99


,check,count
0,Official answer IDs,99
1,Also matching implemented timestamp rule,91
2,Needs timestamp-rule review,8


,Monster Type,P_ID
0,Vampire,P00000775
1,Vampire,P00000806
2,Vampire,P00000843
3,Vampire,P00000974
4,Vampire,P00001103
...,...,...
94,Vampire,P00021323
95,Vampire,P00021355
96,Vampire,P00021362
97,Vampire,P00021609
